# Daglab GraphQL Integration Example

This notebook demonstrates how to use Daglab's GraphQL client to interact with a Dagster instance.

## Setup and Authentication

In [ ]:
import os
import json
import pandas as pd
from datetime import datetime

# Daglab imports
from daglab.helpers.graphql import DagsterClientSync, DagsterClientError
from daglab.helpers.auth import AuthConfig, AuthType
from daglab.validation.security import (
    validate_graphql_query,
    validate_run_config,
    validate_tags
)

# Configuration
DAGSTER_URL = os.getenv("DAGSTER_URL", "http://localhost:3000/graphql")
DAGSTER_TOKEN = os.getenv("DAGSTER_TOKEN")

print(f"Connecting to Dagster at: {DAGSTER_URL}")

In [ ]:
# Setup authentication
if DAGSTER_TOKEN:
    auth_config = AuthConfig.bearer(DAGSTER_TOKEN)
    print("✅ Using bearer token authentication")
else:
    # Try basic auth
    username = os.getenv("DAGSTER_USERNAME")
    password = os.getenv("DAGSTER_PASSWORD")
    
    if username and password:
        auth_config = AuthConfig.basic(username, password)
        print("✅ Using basic authentication")
    else:
        auth_config = AuthConfig(auth_type=AuthType.NONE)
        print("⚠️ No authentication configured")

# Create client
client = DagsterClientSync(
    endpoint=DAGSTER_URL,
    auth_config=auth_config,
    timeout=30.0,
    verify_ssl=True
)

# Test connection
if client.health_check():
    print("✅ Connected to Dagster successfully!")
else:
    print("❌ Failed to connect to Dagster")

## Discover Repositories and Jobs

In [ ]:
# Query for repositories
discover_query = """
query DiscoverRepositories {
    repositoriesOrError {
        ... on RepositoryConnection {
            nodes {
                id
                name
                location {
                    id
                    name
                }
                jobs {
                    id
                    name
                    description
                }
                pipelines {
                    id
                    name
                    description
                }
                schedules {
                    id
                    name
                    cronSchedule
                    pipelineName
                }
                sensors {
                    id
                    name
                    pipelineName
                    status
                }
            }
        }
        ... on PythonError {
            message
            stack
        }
    }
}
"""

# Validate query first
is_valid, error = validate_graphql_query(discover_query)
if not is_valid:
    print(f"❌ Query validation failed: {error}")
else:
    print("✅ Query validated successfully")

# Execute query
try:
    result = client.query(discover_query)
    
    if "repositoriesOrError" in result and "nodes" in result["repositoriesOrError"]:
        repositories = result["repositoriesOrError"]["nodes"]
        print(f"\nFound {len(repositories)} repositories:")
        
        for repo in repositories:
            print(f"\n📦 Repository: {repo['name']}")
            print(f"   Location: {repo['location']['name']}")
            
            jobs = repo.get('jobs', repo.get('pipelines', []))
            if jobs:
                print(f"   Jobs ({len(jobs)}):")
                for job in jobs:
                    print(f"     - {job['name']}")
                    if job.get('description'):
                        print(f"       {job['description']}")
            
            schedules = repo.get('schedules', [])
            if schedules:
                print(f"   Schedules ({len(schedules)}):")
                for schedule in schedules:
                    print(f"     - {schedule['name']} ({schedule['cronSchedule']})")
            
            sensors = repo.get('sensors', [])
            if sensors:
                print(f"   Sensors ({len(sensors)}):")
                for sensor in sensors:
                    print(f"     - {sensor['name']} [{sensor['status']}]")
    
except DagsterClientError as e:
    print(f"❌ Error: {e}")

## View Recent Pipeline Runs

In [ ]:
# Query for recent runs
runs_query = """
query GetRecentRuns($limit: Int!) {
    pipelineRunsOrError(limit: $limit) {
        ... on Runs {
            results {
                id
                runId
                pipelineName
                status
                startTime
                endTime
                tags {
                    key
                    value
                }
                stats {
                    ... on RunStatsSnapshot {
                        stepsSucceeded
                        stepsFailed
                        expectations
                        materializations
                    }
                }
            }
        }
        ... on PythonError {
            message
        }
    }
}
"""

try:
    result = client.query(runs_query, {"limit": 10})
    
    if "pipelineRunsOrError" in result and "results" in result["pipelineRunsOrError"]:
        runs = result["pipelineRunsOrError"]["results"]
        
        if runs:
            # Convert to DataFrame for better visualization
            runs_data = []
            for run in runs:
                runs_data.append({
                    "Run ID": run["runId"][:8] + "...",
                    "Pipeline": run["pipelineName"],
                    "Status": run["status"],
                    "Start Time": datetime.fromtimestamp(float(run["startTime"]) / 1000) if run.get("startTime") else None,
                    "End Time": datetime.fromtimestamp(float(run["endTime"]) / 1000) if run.get("endTime") else None,
                    "Steps Succeeded": run.get("stats", {}).get("stepsSucceeded", 0),
                    "Steps Failed": run.get("stats", {}).get("stepsFailed", 0)
                })
            
            df = pd.DataFrame(runs_data)
            print("\n📊 Recent Pipeline Runs:")
            display(df)
            
            # Show run status distribution
            status_counts = df["Status"].value_counts()
            print("\n📈 Run Status Distribution:")
            for status, count in status_counts.items():
                print(f"   {status}: {count}")
        else:
            print("No recent runs found")
            
except Exception as e:
    print(f"❌ Error: {e}")

## Launch a Pipeline Run

In [ ]:
# First, let's select a job to run
# (You would normally get this from the discovery query above)
SELECTED_JOB = "example_job"  # Replace with actual job name
SELECTED_REPO = "example_repo"  # Replace with actual repo name
SELECTED_LOCATION = "example_location"  # Replace with actual location name

# Prepare run configuration
run_config = {
    "ops": {},
    "resources": {}
}

# Prepare tags
tags = {
    "source": "daglab-notebook",
    "user": os.getenv("USER", "unknown"),
    "timestamp": datetime.now().isoformat()
}

# Validate configuration
config_valid, config_error = validate_run_config(run_config)
tags_valid, tags_error = validate_tags(tags)

if not config_valid:
    print(f"❌ Config validation failed: {config_error}")
elif not tags_valid:
    print(f"❌ Tags validation failed: {tags_error}")
else:
    print("✅ Configuration validated successfully")
    print(f"\nRun Config: {json.dumps(run_config, indent=2)}")
    print(f"\nTags: {json.dumps(tags, indent=2)}")

In [ ]:
# Launch the job (uncomment to actually run)
"""
launch_mutation = """
mutation LaunchJob($executionParams: ExecutionParams!) {
    launchPipelineExecution(executionParams: $executionParams) {
        __typename
        ... on LaunchRunSuccess {
            run {
                id
                runId
                status
                pipelineName
            }
        }
        ... on RunConfigValidationInvalid {
            errors {
                message
                path
            }
        }
        ... on PythonError {
            message
            stack
        }
    }
}
"""

variables = {
    "executionParams": {
        "selector": {
            "repositoryLocationName": SELECTED_LOCATION,
            "repositoryName": SELECTED_REPO,
            "pipelineName": SELECTED_JOB
        },
        "runConfigData": run_config,
        "mode": "default",
        "executionMetadata": {
            "tags": [{"key": k, "value": str(v)} for k, v in tags.items()]
        }
    }
}

try:
    result = client.mutate(launch_mutation, variables)
    
    launch_result = result.get("launchPipelineExecution", {})
    
    if launch_result.get("__typename") == "LaunchRunSuccess":
        run = launch_result["run"]
        print(f"✅ Job launched successfully!")
        print(f"   Run ID: {run['runId']}")
        print(f"   Status: {run['status']}")
        print(f"   View at: {DAGSTER_URL.replace('/graphql', '')}/instance/runs/{run['runId']}")
    else:
        print(f"❌ Failed to launch job: {launch_result}")
        
except Exception as e:
    print(f"❌ Error: {e}")
"""

## Query Assets

In [ ]:
# Query for assets
assets_query = """
query GetAssets($limit: Int!) {
    assetsOrError(limit: $limit) {
        ... on AssetConnection {
            nodes {
                key {
    }
                description
                repository {
                    name
                }
                definition {
                    opNames
                    computeKind
                }
                latestMaterialization {
                    timestamp
                    runId
                }
            }
        }
        ... on PythonError {
            message
        }
    }
}
"""

try:
    result = client.query(assets_query, {"limit": 20})
    
    if "assetsOrError" in result and "nodes" in result["assetsOrError"]:
        assets = result["assetsOrError"]["nodes"]
        
        print(f"\n📦 Found {len(assets)} assets:")
        
        assets_data = []
        for asset in assets:
            asset_path = ".".join(asset["key"]["path"])
            repo_name = asset.get("repository", {}).get("name", "unknown")
            
            latest_mat = asset.get("latestMaterialization")
            last_updated = None
            if latest_mat and latest_mat.get("timestamp"):
                last_updated = datetime.fromtimestamp(float(latest_mat["timestamp"]))
            
            assets_data.append({
                "Asset": asset_path,
                "Repository": repo_name,
                "Description": asset.get("description", "")[:50] + "..." if asset.get("description") else "",
                "Compute Kind": asset.get("definition", {}).get("computeKind", ""),
                "Last Updated": last_updated
            })
        
        df = pd.DataFrame(assets_data)
        display(df)
        
except Exception as e:
    print(f"❌ Error: {e}")

## Monitor Run Progress

In [ ]:
# Function to monitor a run
def monitor_run(run_id: str, poll_interval: int = 5, max_polls: int = 60):
    """Monitor a run until completion."""
    import time
    
    query = """
    query GetRunStatus($runId: ID!) {
        pipelineRunOrError(runId: $runId) {
            ... on Run {
                id
                status
                startTime
                endTime
                stats {
                    ... on RunStatsSnapshot {
                        stepsSucceeded
                        stepsFailed
                        expectations
                        materializations
                    }
                }
            }
            ... on RunNotFoundError {
                message
            }
        }
    }
    """
    
    terminal_states = ["SUCCESS", "FAILURE", "CANCELED"]
    polls = 0
    
    print(f"Monitoring run {run_id}...")
    
    while polls < max_polls:
        try:
            result = client.query(query, {"runId": run_id})
            
            if "pipelineRunOrError" in result:
                run = result["pipelineRunOrError"]
                
                if "status" in run:
                    status = run["status"]
                    stats = run.get("stats", {})
                    
                    print(f"\r[{datetime.now().strftime('%H:%M:%S')}] Status: {status} | "
                          f"Steps: {stats.get('stepsSucceeded', 0)}/{stats.get('stepsFailed', 0)} | "
                          f"Materializations: {stats.get('materializations', 0)}", end="")
                    
                    if status in terminal_states:
                        print(f"\n\n✅ Run completed with status: {status}")
                        return run
                else:
                    print(f"\n❌ Run not found: {run.get('message', 'Unknown error')}")
                    return None
            
            time.sleep(poll_interval)
            polls += 1
            
        except Exception as e:
            print(f"\n❌ Error monitoring run: {e}")
            return None
    
    print(f"\n⏱️ Monitoring timeout after {max_polls * poll_interval} seconds")
    return None

# Example usage (replace with actual run ID)
# monitor_run("your-run-id-here")

## Cleanup

In [ ]:
# Close the client connection
client.close()
print("✅ Client connection closed")